## Always initiate experiment here!

Important dependencies

In [12]:
%load_ext autoreload
%autoreload 2
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import interpolate
import h5py
import pandas as pd
import kaleido # need pip install kaleido==0.1.0post1 for write_image to work
import re   
import os
import sys 
import time
import datetime

# Import the necessary class and functions from your module
from bender_functions import Bender
bender = Bender()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Where do you want to save the data files?

In [13]:
# Output h5 file (this is the raw data file!). 
outputfile = r'C:\Users\jimen\desktop\BenderData\Mar5_CodeTest_.h5'
outputfile = bender.increment_file_name(outputfile)

# Output an image file of your raw data (torque versus time). This is great for quality control.
outputfig = r'C:\Users\jimen\desktop\BenderData\Mar5_CodeTest_.png'
outputfig = bender.increment_file_name(outputfig)

# Double check output file names
print(outputfig)
print('Actual output file: {}'.format(outputfile))

C:\Users\jimen\desktop\BenderData\Mar5_CodeTest_001.png
Actual output file: C:\Users\jimen\desktop\BenderData\Mar5_CodeTest_001.h5


## Semi-permanent settings. Check these well before an experiment

In [14]:
# Motor/stimulator directionality. This is SUPER important This couldn't shouldn't need to change often, but double check before each experiment!
bender.loadCalibration('FT56491.cal') # Load sensor calibration file. Make sure the max range and sensitivity are appropriate for the size of the animal being used!
positive_motor_direction = "left"     # Does a positive angle command on the motor make the bender go left or right? Check before each experiment!!! Depends on mounting and motor software settings.
bending_axis_sensor = "x"
bending_axis_specimen = "dorsovventral" # How is the specimen being bent? "dorsoventral" or "lateral" or "anteroposterior"
S1side = 'left' # Double check stimulator channel 1 side before each experiment!!!
S2side = 'right'

# DAQ and motor parameters.
samplefreq = 1000.0 # DAQ sample frequency
outputfreq = 100000.0 # DAQ output frequency
device_name = '/Dev1' # DAQ Device name
stepsperrev = 1600      # How many motor steps (signals) per revolution of the motor shaft? (e.g., 200 for 1.8 degree stepper with no microstepping, 1600 for 1.8 degree stepper with 1/8 microstepping)

# Time buffers for stimulation and bending
waitbefore = 3.0 # How many seconds to wait before starting the bending after stimulation starts?
waitafter = 4.0 # How many seconds to wait after ... ?
rampdur = 0.25 # How many seconds to ramp on or off the motor motion?

amp_step_vel = 10 # Double check this
encoder_counts_per_rev = 10000 # External encoder counts per revolution (e.g., 10000 for US Digital E5 optical encoder with 1000 PPR and 1/10 microstepping)
encoder_chan = "ctr0" # Encoder channel on DAQ device

# Stimulation timing parameter for isometric tests. This likely does not need to be changed.
prepoststim_dur = 0.3 / 5       # duty of 0.3 at 5 Hz
prepoststim_sep = 1             # time between left and right bursts
prestim_time = -2           # time prestim left burst starts
poststim_time = 2           # time *after* end of bending

## Biometrics: Input these before mounting

In [15]:
fishcode = "foam" 
segment = "anterior"
fishmass = 100    # Body mass in grams
fishlen_TL = 186     # Total length in mm
fishlen_SL = 185 # Standard length in mm

## Mount-specific dimensions: Input these after mounting and before bending

In [16]:
xsec_width = 10        # mm Cross sectional width of fish between the clamps at the axis of rotation
xsec_height = 83       # mm Cross sectional height of fish between the clamps at the axis of rotation

# Mounted biometrics
dbend = 123     # mm Distance from snout to the center of pressure ?
dclamp = 10     # mm Distance between the two clamps
dvert = 190         # mm Vertical distence from the transducer to the center of pressure
dhoriz = 10          # mm Horizontal distence from the transducer to the center of pressure

## Experimental parameters! Decide on these well before an experiment

If you want to drive curvature based on body thickness and red muscle strain, a simple combination can be performed to create inputs for all_curves.


In [17]:
test_type = "dynamic" # Select your test type!

# Bender specimen parameters
all_freqs = [1]
all_curves = [2]
randomize = False 
cycles_per_step = 5    # How many times do you want to bend your specimen at each amplitude/frequency?
n_end_cycles = 2        # add cycles after last amplitude step
stim_cycles_in_step = np.array([2,3]) # This array defines which cycles to activate

# Bender muscle stimulation parameters
is_stim = False # True stimulates the muscle. False is for passive tests
all_stimduties = [0]       # fractions of a cycle
all_stimphases = [0]      # fractions of a cycle
stim_pulse_rate = 75 # Hz shouldn't have to change this from 75

# Change these to whatever the stimulator panel is set to.
S1volts = 10 # Volts
S2volts = 10 # Volts
S1pulsedur = 2          # ms
S2pulsedur = 2          # ms

# FREQUENCY SWEEP SETTINGS #
duration = 60      # sec. How long to you want the whole test to last?
amplitude_frequency_exponent = 0   # should be between -1 and 0. Zero is constant amplitude, -1 is constant velocity, -0.5 is right in the middle.


## Configure all settings to Bender

In [18]:
# Configure the bender settings 
bender.loadCalibration('FT56491.cal')
bender.set_stim_channels('ao0', 'ao1')
bender.set_motor_channel('port0')
bender.set_encoder_channel(encoder_chan, counts_per_rev=encoder_counts_per_rev)

# Set global parameters
bender.all_curves = all_curves
bender.all_freqs = all_freqs
bender.all_stimphases = all_stimphases
bender.all_stimduties = all_stimduties
bender.is_stim = is_stim

bender.dclamp = dclamp
bender.xsec_width = xsec_width

bender.stepsperrev = stepsperrev
bender.waitbefore = waitbefore
bender.waitafter = waitafter
bender.rampdur = rampdur
bender.amp_step_vel = amp_step_vel
bender.samplefreq = samplefreq
bender.outputfreq = outputfreq

bender.stim_pulse_rate = stim_pulse_rate 
bender.prepoststim_dur = prepoststim_dur
bender.prepoststim_sep = prepoststim_sep
bender.prestim_time = prestim_time
bender.poststim_time = poststim_time


In [19]:
print(all_freqs[-1])

1


## Generate Cycle-Stimulation sequences

In [20]:
# Run the calculation method to generate the sequence attributes. This applies only to some test_types (dynamic, static)
if test_type in ['dynamic', 'static']:
    bender.organize_cycles(
        all_curves=all_curves,
        all_freqs=all_freqs,
        randomize=randomize,
        cycles_per_step=cycles_per_step,
        n_end_cycles=n_end_cycles,
        dclamp=dclamp,
        xsec_width=xsec_width,
        stim_cycles_in_step=stim_cycles_in_step,
        all_stimduties=all_stimduties,
        all_stimphases=all_stimphases,
        stim_pulse_rate=stim_pulse_rate
    )
if test_type in ['sweep']:
    bender.duration = duration # duration of each sweep in seconds
    bender.all_freqs = all_freqs # frequencies of sweep in Hz (should be a list of 2 values: [start, end])
    bender.all_curves = all_curves # amplitude of sweep in degrees
    bender.xsec_width = xsec_width
    bender.amplitude_frequency_exponent = amplitude_frequency_exponent # exponent for how amplitude changes

    

organize_cycles took 0.003147125244140625 seconds


In [ ]:
# Please specify what needs to be fixed or provide the code/error you want help with.
# If you have an error or issue in a specific cell, paste it here for targeted assistance.

## CHECK: cycle parameters and muscle strains

In [ ]:
# CHECK CYCLE PARAMETERS
print("--- Generated Sequence Attributes ---")
print(f"Total move duration (np.sum(period_by_cycle)): {np.sum(bender.period_by_cycle):.2f} seconds")
print(f"Total cycles generated: {len(bender.period_by_cycle)}")

# Print the full arrays for review if they are short enough
print("\nall_freqs by cycle (Hz):")
print(bender.freq_by_cycle)

print("\nAmplitudes by cycle (degrees):")
print(bender.amp_by_cycle)

print("\nDuties cycle (degrees):")
print(bender.duty_by_cycle)

print("\nPhases cycle (degrees):")
print(bender.phase_by_cycle)

print("\nAmplitudes by cycle (degrees):")
print(bender.amp_by_cycle)

print("\nStimulation Burst Durations:")
print(bender.actburstdur)

print(f"\nall_freqs (Start/End Freqs): {bender.all_freqs}")
print(f"allamps (Start/End Amps): {bender.allamps}")

# You can also use a print statement right in the function if you prefer immediate feedback:
# print(f"DEBUG: bender attributes populated successfully in notebook.")

# Check strains and strain rates
pd.DataFrame({"freq (Hz)": bender.all_freqs, 
              "curve (1/m)": bender.all_curves, 
              "amp (deg)": bender.allamps, 
              "strain (%)": bender.allstrains*100,
            "strain rate (%/s)": bender.allstrainrates*100})

--- Generated Sequence Attributes ---
Total move duration (np.sum(period_by_cycle)): 7.60 seconds
Total cycles generated: 38

Frequencies by cycle (Hz):
[5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5.
 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5. 5.]

Amplitudes by cycle (degrees):
[1.14591559 1.14591559 1.14591559 1.14591559 1.14591559 1.14591559
 1.14591559 1.14591559 1.14591559 1.14591559 1.14591559 1.14591559
 1.14591559 1.14591559 1.14591559 1.14591559 1.14591559 1.14591559
 1.14591559 1.14591559 1.14591559 1.14591559 1.14591559 1.14591559
 1.14591559 1.14591559 1.14591559 1.14591559 1.14591559 1.14591559
 1.14591559 1.14591559 1.14591559 1.14591559 1.14591559 1.14591559
 1.14591559 1.14591559]

Duties cycle (degrees):
[0.3 0.3 0.3 0.3 0.3 0.3 0.3 0.3 0.3 0.3 0.3 0.3 0.3 0.3 0.3 0.3 0.3 0.3
 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4
 0.4 0.4]

Phases cycle (degrees):
[-0.13 -0.13 -0.13 -0.13 -0.13 -0.13 -0.13 -0.13 -0.13 -0.5  -0.5 

,freq (Hz),curve (1/m),amp (deg),strain (%),strain rate (%/s)
0,5,2,1.145916,1.0,31.415927
1,5,2,1.145916,1.0,31.415927
2,5,2,1.145916,1.0,31.415927
3,5,2,1.145916,1.0,31.415927


## CONFIGURE DURING: Input dimensions for calculating Moment of Inertia for Clamps and Specimen

In [130]:
# NOTE: All units must now be in millimeters (mm) and grams (g)

# Clamp Dimensions (mm) and Material Properties: H=up/down, W=left/right, D=front/back. This is dimension for a single clamp!
CLAMP_H = 100 
CLAMP_W = 50   
CLAMP_D = 20  
RHO_CLAMP = 0.001116 # Effective density for 90% infill 3D printed PLA (g/mm^3)


# Specimen Dimensions and Material Properties #
# Max dimensions for bounding box of specimen mounted within the clamps. The max is arbitrary, so it can be larger than the actual object. Larger just means more computing time
SPECIMEN_H = 500 # Height of the specimen (up to down, along Y axis)
SPECIMEN_D = 500 # Depth of the specimen (front to back, along Z axis)

# Front Oval (at D=0, the front) dimensions (Total Height x Total Width) measured at the axis of rotation (shaft) to the end of the thick part of the fish (can ignore fin rays)
FRONT_H = 50.0 
FRONT_W = 30.0  
BACK_H = 20.0   
BACK_W = 10.0   

# Density for the mounted specimen inner object (g/mm^3)
RHO_OBJECT = 0.001   

# System Offsets (Distance from the Global Axis of Rotation, which is the Y-axis where W = 0 and D = 0)
SPECIMEN_CM_OFFSET_W = 0.0 # Specimen center is at W=0
SPECIMEN_CM_OFFSET_D = 0.0 # Specimen center is at D=0

# Clamps are positioned symmetrically left/right along the Width (X) axis
# CM_W_Position = (Center of Specimen Width) + (Gap Distance) + (Half Clamp Width)
CLAMP_LEFT_CM_W = - (FRONT_W / 2 + CLAMP_W / 2) # Example positioning
CLAMP_RIGHT_CM_W =   (FRONT_W / 2 + CLAMP_W / 2)
CLAMP_CM_D = (dclamp / 2) + (CLAMP_W / 2) # How far are the centers of the clamps to the axis of rotation?

# How many samples to use for numerical integration (more samples = more accurate, but slower computation)
NUM_SAMPLES = 20 

## CHECKPOINT: Calculate Moments of Inertia for clamp and specimen!

In [131]:
print(f"--- Calculating Clamp (Outer) MOI ---")

# Calculate MOI for Left and Right Clamps
I_clamp_left, M_clamp_left = bender.calculate_moi_clamp(
    H=CLAMP_H, W=CLAMP_W, D=CLAMP_D, rho=RHO_CLAMP, 
    offset_x=CLAMP_LEFT_CM_W, offset_z=CLAMP_CM_D
)
I_clamp_right, M_clamp_right = bender.calculate_moi_clamp(
    H=CLAMP_H, W=CLAMP_W, D=CLAMP_D, rho=RHO_CLAMP, 
    offset_x=CLAMP_RIGHT_CM_W, offset_z=CLAMP_CM_D
)

print(f"Mass of single clamp: {M_clamp_left:.2f} g")
print(f"MOI of two clamps combined: {I_clamp_left + I_clamp_right:.2f} g*mm²")
print("-" * 30)

print(f"--- Calculating Specimen (Inner) MOI ---")

# Convert total dimensions to semi-dimensions for the function inputs
I_specimen, M_specimen = bender.calculate_moi_specimen(
    rho_eff=RHO_OBJECT,
    obj_depth_length=SPECIMEN_D,
    front_h_semi=FRONT_H / 2,
    back_h_semi=BACK_H / 2,
    front_w_semi=FRONT_W / 2,
    back_w_semi=BACK_W / 2,
    num_samples=NUM_SAMPLES, 
    axis_offset_x=SPECIMEN_CM_OFFSET_W, 
    axis_offset_z=SPECIMEN_CM_OFFSET_D
)

print(f"Estimated Mass of specimen: {M_specimen:.2f} g")
print(f"MOI of specimen: {I_specimen:.2f} g*mm²")
print("-" * 30)

# --- 5. Total System MOI ---
I_total_system = I_clamp_left + I_clamp_right + I_specimen # These will be used to calculate torque (Torque = MOI (or I) * angular acceleration)
Total_Mass_System = M_clamp_left + M_clamp_right + M_specimen # Total mass is here to help give us a sense of whether our MOI estimates are reasonable

print(f"Total System Mass: {Total_Mass_System:.2f} g")
print(f"Total System Moment of Inertia (around global axis): {I_total_system:.2f} g*mm²")

--- Calculating Clamp (Outer) MOI ---
Mass of single clamp: 111.60 g
MOI of two clamps combined: 611940.00 g*mm²
------------------------------
--- Calculating Specimen (Inner) MOI ---
Estimated Mass of specimen: 405.63 g
MOI of specimen: 19372887.02 g*mm²
------------------------------
Total System Mass: 628.83 g
Total System Moment of Inertia (around global axis): 19984827.02 g*mm²


## CALCULATE: Get important measurements to include in the file (H5) to be saved. Theoretically not necessary since most values are saved, but simplifies pipeline. 

In [132]:
test_section_pos = dbend/(fishlen_TL + dclamp + CLAMP_D)   # Position of the bending segment at the axis of rotation (fraction of total length). Assuming the clamp depth (CLAMP_D below) is 20. 

# Determine which cycles will have muscle stim

Six channels from the force transducer, plus the monitor channel from the S88 stimulator.

In [1]:
SG0_chan = 'ai0'
SG1_chan = 'ai1'
SG2_chan = 'ai2'
SG3_chan = 'ai3'
SG4_chan = 'ai4'
SG5_chan = 'ai5'

stim_monitor_chan = 'ai6'

inchannels = [SG0_chan, SG1_chan, SG2_chan, SG3_chan, SG4_chan, SG5_chan,
                stim_monitor_chan]
inchannel_names = ['SG0', 'SG1', 'SG2', 'SG3', 'SG4', 'SG5',
                    'stim_monitor']

bender.set_input_channels(inchannels, inchannel_names)

NameError: name 'bender' is not defined

# Plot angles and pulses to CHECK

In [ ]:
fig = make_subplots(rows = 2, cols = 1,
                   shared_xaxes=True)
fig.add_trace(
    go.Scatter(x = t, y = angle, mode="lines", name="angle"),
    row=1, col=1)

for onoff in Lonoff:
    fig.add_vrect(x0 = onoff[0], x1=onoff[1], fillcolor="black", opacity=0.25, line_width=0,
                      row=1, col=1)

for onoff in Ronoff:
    fig.add_vrect(x0 = onoff[0], x1=onoff[1], opacity=0.7, line_width=1,
                      row=1, col=1)

fig.update_yaxes(title_text = "angle (deg)", row=1)
fig.add_trace(
    go.Scatter(x = t, y = anglevel, mode="lines", name="anglevel"),
    row=2, col=1)

fig.update_yaxes(title_text = "angular velocity (deg/s)", row=2)
fig.update_xaxes(title_text = "time (s)", row=2)

NameError: name 't' is not defined

# START BENDING!! 

This is the main code block that runs the experiment. It sets up the DAQ, sends the output, records the input, and writes it to the file.

In [ ]:
aidata = bender.run(device_name)

In [ ]:
forcetorque = bender.applyCalibration(aidata)
forcetorque_names = ['xForce', 'yForce', 'zForce', 'xTorque', 'yTorque', 'zTorque']

In [ ]:
angle_measured = bender.angle # Need to make sure to add angles for the internal motor encoder, the command angle, and the measured angle from the external encoder

In [ ]:
with h5py.File(outputfile, 'w') as f:

    # --- Group 1: General Experiment Information ---
    g_info = f.create_group('ExperimentInfo')
    g_info.attrs['EndTime'] = bender.endTime.strftime('%Y-%m-%d %H:%M:%S %Z')
    g_info.attrs['FishCode'] = fishcode
    g_info.attrs['Segment'] = segment

    # --- Group 2: Specimen Geometry and Setup ---
    g_specimen = f.create_group('Biometrics')
    g_specimen.attrs['FishLength_mm'] = fishlen
    g_specimen.attrs['FishMass_g'] = fishmass
    g_specimen.attrs['FishCrossSectionWidth_mm'] = xsec_width
    g_specimen.attrs['FishCrossSectionHeight_mm'] = xsec_height
    g_specimen.attrs['TestSectionPosition_perc'] = test_section_pos

    # --- Group 3: Mount Geometry ---
    g_mount = f.create_group('MountGeometry')
    g_mount.attrs['BendLocation_mm'] = dbend
    g_mount.attrs['ClampDistance_mm'] = dclamp
    g_mount.attrs['DistanceFromTransducerVert_mm'] = dvert
    g_mount.attrs['DistanceFromTransducerHoriz_mm'] = dhoriz

    # --- Group 4: Inertial/Mass Properties (MOI) ---
    g_moi = f.create_group('InertialProperties')
    g_moi.attrs['TotalSystemMass_g'] = Total_Mass_System
    g_moi.attrs['TotalSystemMOI_gmm2'] = I_total_system

    # --- Group 5: Physical Dimensions used for calculations ---
    g_dims = f.create_group('CalculationDimensionsMOI')
    g_dims.attrs['Clamp_Height_mm'] = CLAMP_H
    g_dims.attrs['Clamp_Width_mm'] = CLAMP_W
    g_dims.attrs['Clamp_Depth_mm'] = CLAMP_D
    g_dims.attrs['Clamp_Density_gmm3'] = RHO_CLAMP
    g_dims.attrs['Specimen_Height_mm'] = SPECIMEN_H
    g_dims.attrs['Specimen_Depth_mm'] = SPECIMEN_D
    g_dims.attrs['Specimen_FrontHeight_mm'] = FRONT_H
    g_dims.attrs['Specimen_FrontWidth_mm'] = FRONT_W
    g_dims.attrs['Specimen_BackHeight_mm'] = BACK_H
    g_dims.attrs['Specimen_BackWidth_mm'] = BACK_W
    g_dims.attrs['Specimen_Density_gmm3'] = RHO_OBJECT 

    # Start saving raw data, calibrated data, and output data
    gin = f.create_group('RawInput')
    gin.attrs['SampleFrequency'] = samplefreq
  
    # Store measurements
    gin.create_dataset('forcetransducer', data=aidata[:6,:])
    gin.create_dataset('Stimulation_monitor', data=aidata[6,:])

    gcal = f.create_group('Calibrated')
    for ft1, name1 in zip(forcetorque, forcetorque_names):
        gcal.create_dataset(name1, data=ft1)
    gcal.create_dataset('CalibrationMatrix', data=bender.calibration) # Save the calibration matrix used

    ds = gcal.create_dataset('Encoder', data=bender.angledata) # DOUBLE CHECK THIS
    ds.attrs['CountsPerRev'] = encoder_counts_per_rev

    # save the output data
    gout = f.create_group('Output')
    gout.attrs['SampleFrequency'] = outputfreq
    gout.create_dataset('DigitalOut', data=dig)
    gout.create_dataset('SyncInTrainDur', data=S1actcmd)
    gout.create_dataset('SyncInS2Del', data=S2actcmd)
    gout.attrs['S1side'] = S1side
    gout.attrs['S2side'] = S2side
    gout.attrs['S1volts'] = S1volts
    gout.attrs['S2volts'] = S2volts
    gout.attrs['S1pulsedur_ms'] = S1pulsedur    
    gout.attrs['S2pulsedur_ms'] = S2pulsedur    
    
    # Save stimulus parameters
    gout = f.create_group('NominalStimulus')
    gout.attrs['Type'] = 'Dynamic'

    gout.create_dataset('t', data=t)
    ds = gout.create_dataset('Position', data=angle)
    ds.attrs['Units'] = 'deg'
    ds = gout.create_dataset('Velocity', data=anglevel)
    ds.attrs['Units'] = 'deg/sec'
    gout.create_dataset('tnorm', data=tnorm)
    gout.create_dataset('Lonoff', data=Lonoff)
    gout.create_dataset('Ronoff', data=Ronoff)

    # Save bending parameters
    # Think about how to save all amps, curves, strains, velocities, etc.
    all_amps = []  # TODO: Replace with actual amplitude data if available
    gout.attrs['Curvatures'] = all_curves
    gout.attrs['Frequencies'] = all_freqs
    gout.attrs['CyclesPerStep'] = cycles_per_step
    gout.attrs['EndCycles'] = n_end_cycles
    gout.attrs['FrequencyByCycle'] = freq_by_cycle
    gout.attrs['AmplitudeByCycle'] = amp_by_cycle
    gout.attrs['IsStimByCycle'] = is_stim_cycle
    gout.attrs['CycleRandomOrder'] = order
    gout.attrs['MovementDuration'] = movedur

    # Add estimated red muscle strain and strain rate
    gout.attrs['Strains'] = allstrains
    gout.attrs['StrainRates'] = allstrainrates  

    gout.attrs['WaitPre'] = waitbefore
    gout.attrs['WaitPost'] = waitafter
    gout.attrs['PrePostStimDur'] = prepoststim_dur
    gout.attrs['ScaleFactor'] = scale
    gout.attrs['PositiveMotorDirection'] = positive_motor_direction

    gout.attrs['StimulationOn'] = is_stim
    gout.attrs['StimulationDuty'] = all_stimduties
    gout.attrs['StimulationPhase'] = all_stimphases
    gout.attrs['StimulationPulseRate'] = stim_pulse_rate


##  Zero the data for plotting results

In [ ]:
for ft1 in forcetorque:
    ft1 -= np.mean(ft1[t < 0])

## QUALITY CONTROL: Plot control signals AND raw torque over time

In [ ]:
# Make sure to plot the correct bending axis. It depends on sensor mounting and orientation!
fig = make_subplots(rows = 4, cols = 1,
                   shared_xaxes=True)
#fig.add_trace(
 #   go.Scatter(x = t, y = angle_measured, mode="lines", name="angle_enc"),
  #  row=1, col=1)
fig.add_trace(
    go.Scatter(x = t, y = angle, mode="lines", name="angle_cmd"),
    row=1, col=1)
fig.add_trace(
    go.Scatter(x = t, y = aidata[6,:], mode="lines", name="stim"),
    row=4, col=1)
fig.add_trace(
    go.Scatter(x = t, y = forcetorque[3,:], mode="lines", name="Tx"),
    row=2, col=1)
# fig.add_trace(
#     go.Scatter(x = t, y = forcetorque[1,:], mode="lines", name="Fy"),
#     row=4, col=1)
fig.add_trace(
    go.Scatter(x = t, y = forcetorque[5,:], mode="lines", name="Tz"),
    row=3, col=1)

for onoff in Lonoff:
    fig.add_vrect(x0 = onoff[0], x1=onoff[1], fillcolor="black", opacity=0.25, line_width=0,
                        row="all", col="all")

for onoff in Ronoff:
    fig.add_vrect(x0 = onoff[0], x1=onoff[1], opacity=0.7, line_width=1,
                      row="all", col="all")
fig.update_yaxes(title_text = "angle (deg)", row=1)
fig.update_yaxes(title_text = "Tx (Nm)", row=2)
fig.update_yaxes(title_text = "Tz (Nm)", row=3)
fig.update_yaxes(title_text = "stim", row=4)
fig.update_xaxes(title_text = "time (s)", row=3)
fig.update_layout(title_text = bender.filename)

# Save the plot in output folder
fig.write_image(outputfig, format= 'png', width=1200, height=800)


## QUALITY CONTROL: Do the loops like nice?

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=angle, y=forcetorque[3,:]))
fig.update_yaxes(title_text="torque (Nm)")
fig.update_xaxes(title_text="angle (deg)")